# Modul

In [42]:
import re
import string
import pandas as pd
import csv
import requests

from io import StringIO
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

%pip install python-Levenshtein
%pip install --upgrade git+https://github.com/ariaghora/mpstemmer.git
from mpstemmer import MPStemmer

!pip install Sastrawi
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


  Cloning https://github.com/ariaghora/mpstemmer.git to /tmp/pip-req-build-r2khukau
  Running command git clone --filter=blob:none --quiet https://github.com/ariaghora/mpstemmer.git /tmp/pip-req-build-r2khukau
  Resolved https://github.com/ariaghora/mpstemmer.git to commit 25a5fd923af163a7eac3a5ec976984156ca8fa8b
  Preparing metadata (setup.py) ... done


In [43]:
#%pip freeze > requirements.txt

# Preprocessing

In [44]:
app_reviews_df = pd.read_csv('/content/ulasan_grab.csv')

In [45]:
app_reviews_df.head(5)

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,f26cd5af-bb79-4a17-91cc-b4df5f687571,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,tolong perbaiki Maps pada aplikasi. belum belo...,5,252,5.354.0,2025-04-19 11:06:08,NaN,NaN,5.354.0
1,8f23ff8b-a31d-4c83-bfdd-a2e0dd8c05af,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,banyak driver taxi yang mengeluh tetang pembay...,5,140,5.354.0,2025-04-19 02:26:09,Terima kasih atas kepercayaan kakak pada layan...,2025-04-19 04:26:33,5.354.0
2,a81c63d8-387a-4116-9f5d-ac729e9f14da,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"Ada beberapa yang mengganjal, Mohon benahi: 1....",5,2703,5.347.0,2025-03-08 15:50:59,NaN,NaN,5.347.0
3,16b5e7fa-1229-419a-be97-29c1d8e74c9d,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,Di edit jadi bintang 1 dulu ya karena driverny...,1,137,5.350.0,2025-03-31 07:29:10,Maaf ya atas ketidaknyamanan yang terjadi.\nKa...,2025-03-31 07:41:03,5.350.0
4,2aef357d-8106-4416-92d9-f58871413480,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,driver bike/car/food bisa meng-cancel atau men...,1,151,5.346.0,2025-02-28 15:03:04,Halo Kak. Maaf ya buat engga nyaman 🙇‍♀️ Apabi...,2025-02-28 16:08:29,5.346.0


In [46]:
app_reviews_df.describe().T

,count,mean,std,min,25%,50%,75%,max
score,171000.0,3.289456,1.771534,1.0,1.0,4.0,5.0,5.0
thumbsUpCount,171000.0,3.206053,45.598118,0.0,0.0,0.0,0.0,4168.0


In [47]:
clean_df = app_reviews_df.dropna()
clean_df = clean_df.drop_duplicates()
jumlah_ulasan_setelah_hapus_duplikat, jumlah_kolom_setelah_hapus_duplikat = clean_df.shape

In [48]:
app_reviews_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 171000 entries, 0 to 170999
Data columns (total 11 columns):
 #   Column                Non-Null Count   Dtype 
---  ------                --------------   ----- 
 0   reviewId              171000 non-null  object
 1   userName              171000 non-null  object
 2   userImage             171000 non-null  object
 3   content               171000 non-null  object
 4   score                 171000 non-null  int64 
 5   thumbsUpCount         171000 non-null  int64 
 6   reviewCreatedVersion  138866 non-null  object
 7   at                    171000 non-null  object
 8   replyContent          39872 non-null   object
 9   repliedAt             39872 non-null   object
 10  appVersion            138866 non-null  object
dtypes: int64(2), object(9)
memory usage: 14.4+ MB


In [49]:
slangwords = {
    'otw': 'on the way',
    'pas': 'tepat',
    'kzl': 'kesal',
    'brg': 'barang',
    'dtg': 'datang',
    'sygn': 'sayang',
    'hrs': 'harus',
    'tp': 'tapi',
    'kmrn': 'kemarin',
    'kmrn': 'kemarin',
    'dpt': 'dapat',
    'bgs': 'bagus',
    'tdk': 'tidak',
    'brp': 'berapa',
    'gw': 'saya',
    'sis': 'sisa',
    'bkn': 'bukan',
    'kalo': 'kalau',
    'tdk': 'tidak',
    'blm': 'belum',
    'bnyk': 'banyak',
    'sih': 'sih',
    'jg': 'juga',
    'km': 'kilometer',
    'gmn': 'gimana',
    'bro': 'saudara',
    'sis': 'saudari',
    'asap': 'secepat mungkin',
    'cpt': 'cepat',
    'dr': 'dari',
    'kli': 'kali',
    'emg': 'memang',
    'udh': 'sudah',
    'plg': 'pulang',
    'drmn': 'dari mana',
    'aneh': 'aneh',
    'malah': 'malah',
    'jadi': 'jadi',
    'aja': 'saja',
    'dlu': 'dulu',
    'sinyal': 'signal',
    'ngecharge': 'mengisi daya',
    'cr': 'cari',
    'bisa': 'bisa',
    'lho': 'loh',
    'yah': 'yah',
    'banget': 'sekali',
    'loh': 'loh',
    'mlm': 'malam',
    'pagi': 'pagi',
    'siang': 'siang',
    'sore': 'sore',
    'trs': 'terus',
    'sampe': 'sampai',
    'skrg': 'sekarang',
    'btw': 'by the way',
    'punya': 'punya',
    'km': 'kamu',
    'temen': 'teman',
    'dong': 'dong',
    'pdhl': 'padahal',
    'tlp': 'telepon',
    'lg': 'lagi',
    'udh': 'sudah',
    'klr': 'keluar',
    'wkt': 'waktu',
    'dmn': 'di mana',
    'antri': 'antre',
    'lama': 'lama',
    'hrg': 'harga',
    'apk': 'aplikasi',
    'ga': 'tidak',
    'tau': 'tahu',
    'susah': 'susah',
    'mudah': 'mudah',
    'error': 'error',
    'admin': 'admin',
    'cs': 'customer service',
    'top': 'terbaik',
    'mantap': 'hebat',
    'kurir': 'pengantar',
    'tarif': 'tarif',
    'transaksi': 'transaksi',
    'bengkel': 'bengkel',
    'bus': 'bus',
    'mampir': 'singgah',
    'sopir': 'pengemudi',
    'taksi': 'taksi',
    'ojol': 'ojek online',
    'angkot': 'angkutan kota',
    'bensin': 'bahan bakar',
    'ngebut': 'cepat',
    'ngetem': 'menunggu penumpang',
    'pelit': 'kikir',
    'service': 'layanan',
    'promo': 'promosi',
    'cod': 'cash on delivery',
    'respon': 'tanggapan',
    'feedback': 'umpan balik',
    'refund': 'pengembalian dana',
    'voucher': 'kupon',
    'diskon': 'potongan harga',
    'murah': 'murah',
    'mahal': 'mahal',
    'macet': 'macet',
    'jemput': 'menjemput',
    'antar': 'mengantar',
    'tujuan': 'destinasi',
    'bonus': 'bonus',
    'batal': 'membatalkan',
    'delay': 'penundaan',
    'reschedule': 'menjadwal ulang',
    'rating': 'penilaian',
    'review': 'ulasan',
    'komentar': 'komentar',
    'kualitas': 'kualitas',
    'kecewa': 'kecewa',
    'puas': 'puas',
    'rahasia': 'privasi',
    'update': 'pembaruan',
    'lapor': 'melaporkan',
    'refund': 'pengembalian dana',
    'klaim': 'klaim',
    'saldo': 'saldo',
    'rekap': 'rekapitulasi',
    'driver': 'pengemudi',
    'partner': 'mitra',
    'fasilitas': 'fasilitas',
    'kendala': 'hambatan',
    'jemputan': 'penjemputan',
    'tujuan': 'destinasi',
    'order': 'pesanan',
    'pesan': 'memesan',
    'bayar': 'membayar',
    'aman': 'aman',
    'amanah': 'tepercaya',
    'layanan': 'layanan',
    'cepat': 'cepat',
    'praktis': 'praktis',
    'ribet': 'rumit',
    'keluhan': 'keluhan',
    'kelola': 'mengelola',
    'promo': 'promosi',
    'frekuensi': 'jumlah',
    'stabil': 'stabil',
    'akurat': 'tepat',
    'jadwal': 'jadwal',
    'estimasi': 'perkiraan',
    'pengalaman': 'pengalaman',
    'memuaskan': 'memuaskan',
    'klaim': 'klaim',
    'rating': 'penilaian',
    'negatif': 'negatif',
    'positif': 'positif',
    'review': 'ulasan',
    'pembayaran': 'pembayaran',
    'transaksi': 'transaksi',
    'konfirmasi': 'konfirmasi',
    'notifikasi': 'pemberitahuan',
    'user': 'pengguna',
    'komplain': 'keluhan',
    'terima': 'menerima',
    'kirim': 'mengirim',
    'cetak': 'mencetak',
    'logistik': 'pengiriman',
    'kontak': 'menghubungi',
    'resi': 'nomor pengiriman',
    'pelanggan': 'pelanggan',
    'transparan': 'terbuka',
    'penyelesaian': 'solusi',
    'garansi': 'jaminan',
    'pelayanan': 'layanan',
    'responsif': 'cepat tanggap',
    'respons': 'tanggapan',
    'integritas': 'kejujuran',
    'kecepatan': 'kecepatan',
    'pengembalian': 'pengembalian',
    'kompetitif': 'bersaing',
    'kemudahan': 'kemudahan',
    'fleksibel': 'mudah',
    'tersedia': 'tersedia',
    'sesuai': 'cocok',
    'menyenangkan': 'menyenangkan',
    'kekurangan': 'kekurangan',
    'keuntungan': 'keuntungan',
    'kerugian': 'kerugian',
    'keperluan': 'keperluan'
    }

def slangwordsText(text):
    words = text.split()
    words_fixed = [slangwords[word.lower()] if word.lower() in slangwords else word for word in words]

    return ' '.join(words_fixed)

def emoji_remove_ml(text):
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"  # Emoticons
        "\U0001F300-\U0001F5FF"  # Symbols & Pictographs
        "\U0001F680-\U0001F6FF"  # Transport & Map
        "\U0001F700-\U0001F77F"  # Alchemical Symbols
        "\U0001F780-\U0001F7FF"  # Geometric Shapes Extended
        "\U0001F800-\U0001F8FF"  # Supplemental Arrows-C
        "\U0001F900-\U0001F9FF"  # Supplemental Symbols & Pictographs
        "\U0001FA00-\U0001FA6F"  # Chess Symbols, Domino Tiles, Mahjong Tiles...
        "\U0001FA70-\U0001FAFF"  # Symbols & Pictographs Extended-A
        "\u2600-\u26FF"          # Miscellaneous Symbols (dingbats)
        "\u2700-\u27BF"          # Dingbats
        "\uFE00-\uFE0F"          # Variation Selectors
        "]+",
        flags=re.UNICODE
    )

    emoticon_pattern = re.compile(
        r'(:\)|:\(|:D|:P|:p|;P|;\)|<3|:\'\(|:\'\))'
    )

    text = emoji_pattern.sub('', text)
    text = emoticon_pattern.sub('', text)
    text = re.sub(r'\s{2,}', ' ', text).strip()
    return text

def cleaningText(text):
    text = re.sub(r'@[A-Za-z0-9]+', '', text) # menghapus mention
    text = re.sub(r'#[A-Za-z0-9]+', '', text) # menghapus hashtag
    text = re.sub(r'RT[\s]', '', text) # menghapus RT
    text = re.sub(r"http\S+", '', text) # menghapus link
    text = re.sub(r'[0-9]+', '', text) # menghapus angka
    text = re.sub(r'[^\w\s]', '', text) # menghapus karakter selain huruf dan angka

    text = text.replace('\n', ' ') # mengganti baris baru dengan spasi
    text = text.translate(str.maketrans('', '', string.punctuation)) # menghapus semua tanda baca
    text = text.strip(' ') # menghapus karakter spasi dari kiri dan kanan teks
    return text

def casefoldingText(text):
    text = text.lower()
    return text

def tokenizingText(text):
    text = word_tokenize(text)
    return text

def filteringText(tokens):
    stopwords_indonesia = set(stopwords.words('indonesian'))
    stopwords_english = set(stopwords.words('english'))
    stopwords_indonesia.update(stopwords_english)
    return [word for word in tokens if word not in stopwords_indonesia]

def stemmingText(text):
    stemmer = MPStemmer()
    return ' '.join(stemmer.stem(word) for word in text.split())

def toSentence(list_words):
    sentence = ' '.join(word for word in list_words)
    return sentence


def combineFunction(text):
    text = cleaningText(text)             # Membersihkan mention, hashtag, link, angka, dll.
    text = emoji_remove_ml(text)          # Menghapus emoji & emoticon
    text = casefoldingText(text)          # Mengubah ke huruf kecil
    text = slangwordsText(text)           # Normalisasi slang kata ML
    text = stemmingText(text)             # Stemming ke bentuk dasar
    tokens = tokenizingText(text)         # Tokenisasi
    tokens = filteringText(tokens)        # Hapus stopwords
    return tokens

In [50]:
clean_df['filtering_text'] = clean_df['content'].apply(combineFunction)

clean_df['content_clean'] = clean_df['filtering_text'].apply(toSentence)

print(len(clean_df))
clean_df.head(2)

30501


,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion,filtering_text,content_clean
1,8f23ff8b-a31d-4c83-bfdd-a2e0dd8c05af,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,banyak driver taxi yang mengeluh tetang pembay...,5,140,5.354.0,2025-04-19 02:26:09,Terima kasih atas kepercayaan kakak pada layan...,2025-04-19 04:26:33,5.354.0,"[kemudi, taxi, keluh, tetang, bayar, harga, ap...",kemudi taxi keluh tetang bayar harga aplikasi ...
3,16b5e7fa-1229-419a-be97-29c1d8e74c9d,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,Di edit jadi bintang 1 dulu ya karena driverny...,1,137,5.350.0,2025-03-31 07:29:10,Maaf ya atas ketidaknyamanan yang terjadi.\nKa...,2025-03-31 07:41:03,5.350.0,"[edit, bintang, iya, drivernya, semenamena, kh...",edit bintang iya drivernya semenamena khusus g...


In [51]:
# kamus untuk kata-kata positif
positive_dictionary = dict()

# import kamus positif dari github, credit to the owner: angelmetanosaa
res = requests.get('https://raw.githubusercontent.com/angelmetanosaa/dataset/main/lexicon_positive.csv')

# logika jika data tidak dapat diambil
if res.status_code == 200:
    reader = csv.reader(StringIO(res.text), delimiter=',')

    # menambahkan kata-kata positif dan skor ke dalam positive_dictionary
    for row in reader:
        positive_dictionary[row[0]] = int(row[1])

else:
    print('Unable to fetch positive lexicon data')

# kamus untuk kata-kata negatif
negative_dictionary = dict()

# import kamus negatif dari github, credit to the owner: angelmetanosaa
res = requests.get('https://raw.githubusercontent.com/angelmetanosaa/dataset/main/lexicon_negative.csv')

# logika jika data tidak dapat diambil
if res.status_code == 200:
    reader = csv.reader(StringIO(res.text), delimiter=',')

    # menambahkan kata-kata negatif dan skor ke dalam negative_dictionary
    for row in reader:
        negative_dictionary[row[0]] = int(row[1])

else:
    print('Unable to fetch negative lexicon data')

In [52]:
def analysis_sentiment(words):
    score = 0;

    for word in words:

        if(word in positive_dictionary):
            score = score + positive_dictionary[word]

        if(word in negative_dictionary):
            score = score + negative_dictionary[word]
    polarity = 0

    if (score > 0):
        polarity = 2 # Positive
    elif(score < 0 ):
        polarity = 0 # Negative
    else:
        polarity = 1 # Netral

    return polarity, score

In [53]:
result = clean_df['filtering_text'].apply(analysis_sentiment)
result = list(zip(*result))

clean_df['polarity'] = result[0]
clean_df['polarity_score'] = result[1]
print(clean_df['polarity'].value_counts())
print('\n')
print(clean_df['content_clean'].head())

polarity
0    18495
2     8877
1     3129
Name: count, dtype: int64


1    kemudi taxi keluh tetang bayar harga aplikasi ...
3    edit bintang iya drivernya semenamena khusus g...
4    kemudi bikecarfood mengcancel tolak enak pesan...
6    tapa iklan video puter otomatis pausestop ikla...
7    aplikasi kemudi nya tunggu gerak tanggap kali ...
Name: content_clean, dtype: object


# Model

In [54]:
X = clean_df['content_clean']
Y = clean_df['polarity']

# memberikan nilai setiap fitur dengan teknik TF-IDF
tf_idf = TfidfVectorizer(max_features=1000, min_df=17, max_df=0.8)
X_tf_idf = tf_idf.fit_transform(X)

# mengubah vektor menjadi Dataframe
features_df = pd.DataFrame(X_tf_idf.toarray(), columns=tf_idf.get_feature_names_out())
features_df

,abang,abis,acara,aco,ad,adil,admin,adu,aduh,ah,...,wifi,wilayah,wktu,wkwk,woi,woy,xl,yaa,yaaa,yah
0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.126294,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30496,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
30497,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
30498,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
30499,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [55]:
X_train, X_test, y_train, y_test = train_test_split(X_tf_idf.toarray(), Y, test_size=0.2, random_state=42)

print('Train Data Shape: ', X_train.shape, y_train.shape)
print('Test Data Shape: ', X_test.shape, y_test.shape)

Train Data Shape:  (24400, 1000) (24400,)
Test Data Shape:  (6101, 1000) (6101,)


## Model 1

In [56]:
# Inisialisasi model SVM
model_1 = SVC(kernel='linear', C=1.0)

# Melatih model SVM
model_1.fit(X_train, y_train)

# Prediksi pada data latih dan data uji
y_pred_train_svm = model_1.predict(X_train)
y_pred_test_svm = model_1.predict(X_test)

# Evaluasi akurasi
accuracy_train_1 = accuracy_score(y_train, y_pred_train_svm)
accuracy_test_1 = accuracy_score(y_test, y_pred_test_svm)

# Menampilkan hasil evaluasi
print('\nTrain Accuracy:', accuracy_train_1)
print('Test Accuracy:', accuracy_test_1)
print('\nClassification Report (Test):\n', classification_report(y_test, y_pred_test_svm))



Train Accuracy: 0.9166393442622951
Test Accuracy: 0.8980495000819538

Classification Report (Test):
               precision    recall  f1-score   support

           0       0.92      0.96      0.94      3720
           1       0.75      0.56      0.64       623
           2       0.89      0.89      0.89      1758

    accuracy                           0.90      6101
   macro avg       0.85      0.80      0.82      6101
weighted avg       0.89      0.90      0.89      6101



## Model 2

In [57]:
# Inisialisasi model Random Forest
model_2 = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

model_2.fit(X_train, y_train)

# Prediksi pada data latih dan data uji
y_pred_train_rf = model_2.predict(X_train)
y_pred_test_rf = model_2.predict(X_test)

# Evaluasi akurasi
accuracy_train_2 = accuracy_score(y_train, y_pred_train_rf)
accuracy_test_2 = accuracy_score(y_test, y_pred_test_rf)

# Menampilkan hasil evaluasi
print('\nTrain Accuracy:', accuracy_train_2)
print('Test Accuracy:', accuracy_test_2)
print('\nClassification Report (Test):\n', classification_report(y_test, y_pred_test_rf))


Train Accuracy: 0.9971311475409836
Test Accuracy: 0.8403540403212588

Classification Report (Test):
               precision    recall  f1-score   support

           0       0.85      0.93      0.89      3720
           1       0.76      0.56      0.64       623
           2       0.83      0.74      0.78      1758

    accuracy                           0.84      6101
   macro avg       0.82      0.74      0.77      6101
weighted avg       0.84      0.84      0.84      6101



## Model 3

In [58]:
# memberikan nilai setiap fitur dengan teknik TF-IDF
tf_idf = TfidfVectorizer(max_features=1000, min_df=17, max_df=0.8)
X_tf_idf = tf_idf.fit_transform(X)


# Split data 70/30
X_train, X_test, y_train, y_test = train_test_split(X_tf_idf, Y, test_size=0.3, random_state=42)

print('Train Data Shape:', X_train.shape, y_train.shape)
print('Test Data Shape:', X_test.shape, y_test.shape)

# Inisialisasi dan latih model Random Forest
model_3 = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model_3.fit(X_train, y_train)

# Prediksi
y_pred_train = model_3.predict(X_train)
y_pred_test = model_3.predict(X_test)

# Evaluasi akurasi
accuracy_train_3 = accuracy_score(y_train, y_pred_train)
accuracy_test_3 = accuracy_score(y_test, y_pred_test)

# Tampilkan hasil
print('\nTrain Accuracy:', accuracy_train_3)
print('Test Accuracy:', accuracy_test_3)
print('\nClassification Report (Test):\n', classification_report(y_test, y_pred_test))


Train Data Shape: (21350, 1000) (21350,)
Test Data Shape: (9151, 1000) (9151,)

Train Accuracy: 0.9971896955503513
Test Accuracy: 0.8406731504753578

Classification Report (Test):
               precision    recall  f1-score   support

           0       0.86      0.94      0.89      5588
           1       0.76      0.56      0.65       940
           2       0.83      0.74      0.78      2623

    accuracy                           0.84      9151
   macro avg       0.81      0.75      0.77      9151
weighted avg       0.84      0.84      0.84      9151



In [59]:
result_df = pd.DataFrame({
    'Model': ['Model_1', 'Model_2', 'Model_3'],
    'Accuracy Train': [accuracy_train_1, accuracy_train_2, accuracy_train_3],
    'Accuracy Test': [accuracy_test_1, accuracy_test_2, accuracy_test_3]
})

accuracy_test_all = result_df[['Model', 'Accuracy Test']]
print(accuracy_test_all.sort_values(by='Accuracy Test', ascending=False))

     Model  Accuracy Test
0  Model_1       0.898050
2  Model_3       0.840673
1  Model_2       0.840354


In [60]:
label_map = {0: "Negatif", 1: "Netral", 2: "Positif"}

def predict_with_all_models(text_input):
    print(f"\nTeks: '{text_input}'")

    vector_1 = tf_idf.transform([text_input]).toarray()

    # Prediksi menggunakan model SVM (Model 1)
    pred_1 = model_1.predict(vector_1)[0]
    print(f"Model 1 (SVM) Prediksi: {label_map.get(pred_1)} (Label: {pred_1})")

    pred_2 = model_2.predict(vector_1)[0]
    print(f"Model 2 (Random Forest) Prediksi: {label_map.get(pred_2)} (Label: {pred_2})")

    pred_3 = model_3.predict(vector_1)[0]
    print(f"Model 3 (Random Forest + TF-IDF) Prediksi: {label_map.get(pred_3)} (Label: {pred_3})")

input_texts = ["saya puas dengan pelayanannya"]

for text in input_texts:
    predict_with_all_models(text)


Teks: 'saya puas dengan pelayanannya'
Model 1 (SVM) Prediksi: Positif (Label: 2)
Model 2 (Random Forest) Prediksi: Positif (Label: 2)
Model 3 (Random Forest + TF-IDF) Prediksi: Positif (Label: 2)
